# 호우날두 — 모델 비교·튜닝 (handong_tuning)
`suah_feature_exp.ipynb`의 STEP 1–4(로드, 사상 ID, 공통 fold, `make_features`, 누수 검증)를 그대로 가져와서 그 위에 모델 비교·튜닝만 추가하는 노트북.

| 실험 | 내용 |
|---|---|
| 기준 | 베이스라인 LightGBM 파라미터 (dev OOF RMSE 0.2182) |
| 1 | LightGBM 튜닝: `min_data_in_leaf` → `num_leaves` → 정규화 순서로 좌표 탐색 |
| 2 | 비교 모델 1개: sklearn `HistGradientBoostingRegressor` (추가 설치 없음) |
| 최종 | 단일 모델 또는 OOF 기반 2-모델 블렌딩 |

**수아 오차 분석 반영 (docs/02 §5)**: 정답 > 0.5 m 행(5.8%)이 오차제곱합의 71.5%이고 큰 상승을 0.5–0.8 m 과소예측. `min_data_in_leaf=200`이 극단값 예측을 뭉갤 수 있어 튜닝 1순위로 둠. 모든 실험에서 전체 RMSE와 함께 **큰 상승·상승 시작 구간의 편향**을 기록해, 이 구간이 줄면서 나머지 구간이 나빠지지 않는지 같이 봄.

**규칙**: fold는 재혁이 확정한 `outputs/common/event_folds.csv`를 읽어서 쓴다. 홀드아웃 사상(fold = -1)은 튜닝에서 제외하고, 최종 조합 확정 후 한 번만 확인. 피처는 수아의 `final_features.json`을 그대로 씀 (피처 효과와 튜닝 효과를 섞지 않음). 목적함수는 MSE 유지.

## STEP 0. 환경 설정

In [ ]:
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.ensemble import HistGradientBoostingRegressor

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

# 레포 루트 또는 notebooks/ 어디서 실행해도 data/를 찾음
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA_DIR = ROOT / "data"
OUT_DIR = DATA_DIR.parent / "outputs"
OUT_DIR.mkdir(exist_ok=True)

SEED = 42
N_FOLDS = 4
TARGET = "target_maxrise_6h"
EXPECTED = {"train_rows": 749_657, "test_rows": 186_263, "train_events": 122, "test_events": 30}

RESULT_DIR = OUT_DIR / "handong_tuning"
RESULT_DIR.mkdir(exist_ok=True)

FOLD_FILE = OUT_DIR / "common" / "event_folds.csv"                 # 재혁 확정 (홀드아웃은 fold = -1)
FEATURE_FILE = OUT_DIR / "feature_exp_v1" / "final_features.json"   # 수아 확정 피처
MIN_GAIN = 0.0005        # 이 이상 RMSE가 줄고
MIN_FOLDS_IMPROVED = 3   # 4개 fold 중 3개 이상에서 개선돼야 채택
BASE_RMSE_REF = 0.2182   # 수아 기준 dev OOF (재현 확인용)

# 수아 오차 분석과 같은 구간 정의
BIG_RISE = 0.5           # 정답 > 0.5 m = 큰 상승
FLAT_DWL = 0.05          # dwl_1h ≤ 0.05 = 아직 정체

## STEP 1. 데이터 로드
`train.csv`·`train.csv.gz` 어느 쪽이 있어도 읽도록 `find_file` 사용.

In [ ]:
def find_file(stem):
    hits = sorted(p for p in DATA_DIR.glob(stem + "*") if not p.name.startswith("~$"))
    if not hits:
        raise FileNotFoundError(f"{stem} 파일을 {DATA_DIR}에서 찾지 못함")
    return hits[0]

def read_table(path):
    if path.suffix in (".xlsx", ".xls"):
        return pd.read_excel(path)
    for enc in ("utf-8-sig", "cp949"):
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"인코딩 확인 필요: {path}")

train = pd.read_csv(find_file("train"))
test = pd.read_csv(find_file("test"))
sub = read_table(find_file("sample_submission"))
ddict = read_table(find_file("data_dictionary"))

for df in (train, test):
    df["timestamp"] = pd.to_datetime(df["timestamp"])

print("train:", train.shape, "| test:", test.shape, "| sub:", sub.shape)
print("행 수 일치:", len(train) == EXPECTED["train_rows"], len(test) == EXPECTED["test_rows"])
print("test 관측소 중 train에 없는 관측소:", len(set(test.station_id) - set(train.station_id)))
ddict

## STEP 2. 호우사상 ID 복원 (수아 노트북과 동일)

In [ ]:
def add_event_id(df, prefix, gap_h=1):
    ts = pd.Series(np.sort(df["timestamp"].unique()))
    event_no = ts.diff().gt(pd.Timedelta(hours=gap_h)).cumsum()
    m = pd.DataFrame({"timestamp": ts, "event_id": prefix + event_no.astype(str).str.zfill(3)})
    return df.merge(m, on="timestamp", how="left")

train = add_event_id(train, "TR_E")
test = add_event_id(test, "TE_E")

print("train 사상 수:", train.event_id.nunique(), "(기대", EXPECTED["train_events"], ")")
print("test  사상 수:", test.event_id.nunique(), "(기대", EXPECTED["test_events"], ")")
# 개수가 다르면 사상 사이 간격이 1시간인 경우가 있다는 뜻 → gap_h 조정 또는 wl 연속성으로 추가 검증 필요

## STEP 3. 공통 fold·홀드아웃 불러오기
사상별 행 수가 공통 파일과 같은지 확인 → 사상 ID 복원 결과가 팀과 같다는 뜻.

In [ ]:
def rmse(y, p):
    return float(np.sqrt(np.mean((np.asarray(y) - np.asarray(p)) ** 2)))

ef = pd.read_csv(FOLD_FILE)
my_counts = train.groupby("event_id").size()
assert my_counts.reindex(ef["event_id"]).values.tolist() == ef["n_rows"].tolist(), "사상 ID 복원 결과가 공통 파일과 다름"

fold_of = dict(zip(ef["event_id"], ef["fold"]))
hold_events = set(ef.loc[ef["fold"] == -1, "event_id"])

is_hold = train["event_id"].isin(hold_events)
print(f"dev 사상 {len(fold_of) - len(hold_events)}개 / 홀드아웃 사상 {len(hold_events)}개")
print(f"dev 행 {(~is_hold).sum():,} / 홀드아웃 행 {is_hold.sum():,}")

## STEP 4. 베이스라인 피처 (수아 노트북과 동일한 함수, 관측소 통계는 dev로만)

In [ ]:
EPS = 1e-6

def make_features(df, station_stats):
    df = df.sort_values(["station_id", "timestamp"]).reset_index(drop=True)
    g = df.groupby(["station_id", "event_id"], sort=False)

    def lag(col, k):
        # 정확히 k시간 전 행일 때만 사용 (결측 시각이 있으면 NaN)
        ok = (df["timestamp"] - g["timestamp"].shift(k)) == pd.Timedelta(hours=k)
        return g[col].shift(k).where(ok)

    # A. 수위 변화
    for h in (1, 3, 6, 12, 24):
        df[f"dwl_{h}h"] = df["wl_t"] - df[f"wl_t_minus_{h}h"]
    df["accel_wl"] = df["dwl_1h"] - (df["wl_t_minus_1h"] - df["wl_t_minus_3h"]) / 2
    wl_l2, wl_l4, wl_l5 = lag("wl_t", 2), lag("wl_t", 4), lag("wl_t", 5)
    past_max = pd.concat([df["wl_t"], df["wl_t_minus_1h"], wl_l2, df["wl_t_minus_3h"], wl_l4, wl_l5], axis=1).max(axis=1)
    df["past_maxrise_6h"] = past_max - df["wl_t_minus_6h"]      # 직전 6시간 실현 상승량 (t 이하 값만)
    df["wl_below_past_max"] = df["wl_t"] - past_max             # 최근 고점 대비 위치

    # B. 강우 이력
    for c in ("imerg_rain_1h", "sfc_rain_1h", "radar_reflectivity_mean", "radar_reflectivity_ge20_fraction"):
        for k in (1, 2, 3):
            df[f"{c}_lag{k}"] = lag(c, k)
    df["imerg_trend_1h"] = df["imerg_rain_1h"] - df["imerg_rain_1h_lag1"]
    df["sfc_trend_1h"] = df["sfc_rain_1h"] - df["sfc_rain_1h_lag1"]
    for src in ("imerg", "sfc"):
        df[f"{src}_front_1_6"] = df[f"{src}_rain_1h"] / (df[f"{src}_rain_6h"] + EPS)
        df[f"{src}_front_6_24"] = df[f"{src}_rain_6h"] / (df[f"{src}_rain_24h"] + EPS)
        df[f"{src}_rain_7_24h"] = df[f"{src}_rain_24h"] - df[f"{src}_rain_6h"]
        df[f"{src}_event_cum"] = g[f"{src}_rain_1h"].cumsum()
        df[f"{src}_event_max1h"] = g[f"{src}_rain_1h"].cummax()
    df["imerg_minus_sfc_6h"] = df["imerg_rain_6h"] - df["sfc_rain_6h"]

    # C. 사상 맥락
    df["event_elapsed_h"] = (df["timestamp"] - df.groupby("event_id")["timestamp"].transform("min")).dt.total_seconds() / 3600
    df["wl_since_event_start"] = df["wl_t"] - g["wl_t"].transform("first")

    # D. 동시각 공간 (같은 사상·같은 시각)
    gt = df.groupby(["event_id", "timestamp"])
    for c in ("imerg_rain_1h", "imerg_rain_6h", "sfc_rain_6h", "dwl_1h", "dwl_3h", "dwl_6h"):
        df[f"xs_mean_{c}"] = gt[c].transform("mean")
    df["xs_rising_frac"] = gt["dwl_1h"].transform(lambda s: (s > 0).mean())
    df["dwl_3h_vs_xs"] = df["dwl_3h"] - df["xs_mean_dwl_3h"]

    # E. 관측소 정규화 (train 통계)
    df = df.merge(station_stats, on="station_id", how="left")
    df["wl_rel"] = (df["wl_t"] - df["st_wl_med"]) / (df["st_wl_iqr"] + EPS)
    for h in (1, 3, 6):
        df[f"dwl_{h}h_scaled"] = df[f"dwl_{h}h"] / (df["st_wl_iqr"] + EPS)
    return df

dev_raw = train[~is_hold]
q = dev_raw.groupby("station_id")["wl_t"].quantile([0.25, 0.5, 0.75]).unstack()
station_stats = pd.DataFrame({
    "station_id": q.index,
    "st_wl_med": q[0.5].values,
    "st_wl_iqr": (q[0.75] - q[0.25]).values,
})

tr = make_features(train, station_stats)
te = make_features(test, station_stats)
BASE_FEATURES = [c for c in tr.columns if c not in {"row_id", "timestamp", "event_id", TARGET}]
print("베이스라인 피처 수:", len(BASE_FEATURES))

### 4-1. 누수 자가검증
각 사상의 마지막 6시간 행을 지우고 피처를 다시 만든 뒤, 남은 행의 값이 같은지 비교.

In [ ]:
end = train.groupby("event_id")["timestamp"].transform("max")
cut = train[train["timestamp"] <= end - pd.Timedelta(hours=6)]
tr_cut = make_features(cut, station_stats).set_index("row_id")
tr_full = tr.set_index("row_id").loc[tr_cut.index]

check_cols = [c for c in tr_cut.columns if c not in ("timestamp", "event_id", "station_id", TARGET)]
diff = {c: int((~np.isclose(tr_full[c].astype(float), tr_cut[c].astype(float), equal_nan=True)).sum())
        for c in check_cols}
leaky = {c: n for c, n in diff.items() if n > 0}
print(f"검사한 피처 {len(check_cols)}개 | 미래 의존 피처:", leaky if leaky else "없음 ✅")

## STEP 5. 고정 조건 · 공통 함수
- 피처: 수아 `final_features.json` (현재 기준 72개). 수아가 새 피처를 채택하면 그 생성 함수를 STEP 4 뒤에 붙이고 이 노트북을 다시 돌림
- `run_cv`는 LightGBM·비교 모델 공통. 모든 결과를 `tuning_log.csv`에 누적 (전체·fold RMSE, 구간별 편향, 실행 시간)

In [ ]:
with open(FEATURE_FILE, encoding="utf-8") as f:
    feat_info = json.load(f)
FEATURES = feat_info["features"]
missing = [c for c in FEATURES if c not in tr.columns]
assert not missing, f"수아 피처 생성 코드가 필요함: {missing}"
print(f"피처 {len(FEATURES)}개 | 수아 채택 묶음: {feat_info['selected_groups'] or '없음 (기준 유지)'}")

cats = pd.CategoricalDtype(sorted(train.station_id.unique()))
tr["station_id"] = tr["station_id"].astype(cats)
te["station_id"] = te["station_id"].astype(cats)

tr_fold = tr["event_id"].map(fold_of)
dev = tr[tr_fold >= 0].reset_index(drop=True)
hold = tr[tr_fold == -1].reset_index(drop=True)
dev_fold = dev["event_id"].map(fold_of).to_numpy()
FOLDS = [(np.where(dev_fold != k)[0], np.where(dev_fold == k)[0]) for k in range(N_FOLDS)]

BASE_PARAMS = dict(
    objective="regression", learning_rate=0.05, num_leaves=63,
    min_data_in_leaf=200, feature_fraction=0.8, bagging_fraction=0.8,
    bagging_freq=1, lambda_l2=1.0, seed=SEED, verbose=-1,
)
assert {k: v for k, v in feat_info["params"].items()} == BASE_PARAMS, "수아 비교 파라미터와 기준 파라미터가 다름"

y_dev = dev[TARGET].to_numpy()
SEG = {
    "big": y_dev > BIG_RISE,                                              # 큰 상승
    "onset": (y_dev > BIG_RISE) & (dev["dwl_1h"].to_numpy() <= FLAT_DWL),  # 정체인데 곧 큰 상승
}
SEG["rest"] = ~SEG["big"]

def seg_stats(oof):
    err = oof - y_dev
    sse = (err ** 2).sum()
    return {
        "big_bias": round(float(err[SEG["big"]].mean()), 3),
        "onset_bias": round(float(err[SEG["onset"]].mean()), 3),
        "big_sse_share": round(float((err[SEG["big"]] ** 2).sum() / sse), 3),
        "onset_sse_share": round(float((err[SEG["onset"]] ** 2).sum() / sse), 3),
        "rest_rmse": round(rmse(y_dev[SEG["rest"]], oof[SEG["rest"]]), 4),
    }

def fit_lgb(params, X_tr, y_tr, X_va=None, y_va=None, n_round=None):
    if n_round is None:   # 검증 fold로 early stopping
        return lgb.train(params, lgb.Dataset(X_tr, y_tr), num_boost_round=5000,
                         valid_sets=[lgb.Dataset(X_va, y_va)],
                         callbacks=[lgb.early_stopping(200, verbose=False)])
    return lgb.train(params, lgb.Dataset(X_tr, y_tr), num_boost_round=n_round)

def to_hgb(X):
    # HGB 범주형은 255개까지라 station_id(465개)는 코드값으로 사용
    X = X.copy()
    X["station_id"] = X["station_id"].cat.codes
    return X

def fit_hgb(params, X_tr, y_tr, *_, **__):
    return HistGradientBoostingRegressor(**params).fit(to_hgb(X_tr), y_tr)

def predict(model, X):
    if isinstance(model, lgb.Booster):
        return model.predict(X, num_iteration=model.best_iteration or None)
    return model.predict(to_hgb(X))

LOG = []

def run_cv(name, params, kind="lgb", features=FEATURES):
    t0 = time.time()
    fit = fit_lgb if kind == "lgb" else fit_hgb
    oof = np.zeros(len(dev))
    fold_rmse, best_iters = [], []
    for i_tr, i_va in FOLDS:
        X_tr, X_va = dev.iloc[i_tr][features], dev.iloc[i_va][features]
        m = fit(params, X_tr, y_dev[i_tr], X_va, y_dev[i_va])
        oof[i_va] = predict(m, X_va)
        fold_rmse.append(rmse(y_dev[i_va], oof[i_va]))
        best_iters.append(m.best_iteration if kind == "lgb" else m.n_iter_)
    res = {"name": name, "kind": kind, "params": params, "features": features, "oof": oof,
           "oof_rmse": rmse(y_dev, oof), "fold_rmse": fold_rmse, "best_iters": best_iters,
           "runtime_sec": round(time.time() - t0, 1), **seg_stats(oof)}
    LOG.append({k: v for k, v in res.items() if k not in ("oof", "features")})
    pd.DataFrame(LOG).to_csv(RESULT_DIR / "tuning_log.csv", index=False, encoding="utf-8-sig")
    print(f"[{name}] OOF {res['oof_rmse']:.4f} | fold {np.round(fold_rmse, 4).tolist()} | "
          f"큰상승 편향 {res['big_bias']:+.3f} | 상승시작 편향 {res['onset_bias']:+.3f} | "
          f"나머지 RMSE {res['rest_rmse']:.4f} | iter {best_iters} | {res['runtime_sec']}s")
    return res

def compare(res, ref):
    gain = ref["oof_rmse"] - res["oof_rmse"]
    folds_up = int(sum(a < b for a, b in zip(res["fold_rmse"], ref["fold_rmse"])))
    return gain, folds_up

def passes(res, ref):
    gain, folds_up = compare(res, ref)
    return gain >= MIN_GAIN and folds_up >= MIN_FOLDS_IMPROVED

## STEP 6. 기준 재현
수아 기준(0.2182)과 ±0.0001 이내여야 같은 조건. 다르면 데이터·fold·라이브러리 버전부터 확인.

In [ ]:
results = {"기준": run_cv("기준", BASE_PARAMS)}
print(f"수아 기준과 차이: {results['기준']['oof_rmse'] - BASE_RMSE_REF:+.4f}")
pd.Series(results["기준"]["oof"], name="oof").to_frame().assign(row_id=dev["row_id"]) \
    .to_csv(RESULT_DIR / "oof_기준.csv", index=False)

## STEP 7. LightGBM 튜닝 (좌표 탐색)
한 번에 하나씩 바꾸고, 각 단계의 최고값을 다음 단계로 넘김. 단계 안에서는 OOF가 가장 낮은 값을 고르고, 최종 채택은 STEP 9에서 **기준 대비** 채택 기준(−0.0005 & 3/4 fold)으로 판단.

| 단계 | 바꾸는 값 | 이유 |
|---|---|---|
| 7-1 | `min_data_in_leaf` 100·50·20 | 한동 요청: 200이 극단값(큰 상승) 예측을 뭉갤 수 있음 |
| 7-2 | `num_leaves` 31·127 | 잎을 줄이면 7-1의 과적합을 상쇄, 늘리면 큰 상승 구간을 더 세밀히 |
| 7-3 | `lambda_l2` 5, `feature_fraction` 0.6 | 잎 크기를 줄인 만큼 정규화 보강 |

fold 하나당 약 30초 기준 전체 약 15–20분.

In [ ]:
TUNE_STEPS = [
    ("min_data_in_leaf", [100, 50, 20]),
    ("num_leaves", [31, 127]),
    ("lambda_l2", [5.0]),
    ("feature_fraction", [0.6]),
]

best = results["기준"]
for key, values in TUNE_STEPS:
    for v in values:
        params = {**best["params"], key: v}
        r = run_cv(f"lgb_{key}={v}", params)
        results[r["name"]] = r
        if r["oof_rmse"] < best["oof_rmse"]:
            best = r
    print(f"  -> {key} 단계 최고: {best['name']} ({best['oof_rmse']:.4f})")

LGB_BEST = best
gain, folds_up = compare(LGB_BEST, results["기준"])
print(f"\nLightGBM 최고: {LGB_BEST['name']} | 기준 대비 {gain:+.4f}, 개선 fold {folds_up}/{N_FOLDS} "
      f"-> {'채택 기준 통과' if passes(LGB_BEST, results['기준']) else '기준 미달'}")
print({k: LGB_BEST["params"][k] for k in ("min_data_in_leaf", "num_leaves", "lambda_l2", "feature_fraction")})

## STEP 8. 비교 모델: HistGradientBoosting
- 추가 설치 없이 쓸 수 있는 sklearn 모델. 블렌딩 상대로 쓰려면 단독 성능보다 **LightGBM과 오차가 얼마나 다른지**(OOF 상관)가 중요
- sklearn 내장 early stopping은 행 단위 무작위 검증이라 사상 단위 규칙과 맞지 않음 → 끄고 반복 수 고정
- `min_samples_leaf`는 7-1 최고값을 따라감

In [ ]:
HGB_PARAMS = dict(
    learning_rate=0.05, max_iter=800, max_leaf_nodes=63,
    min_samples_leaf=LGB_BEST["params"]["min_data_in_leaf"],
    l2_regularization=1.0, early_stopping=False, random_state=SEED,
)
results["hgb"] = run_cv("hgb", HGB_PARAMS, kind="hgb")
corr = np.corrcoef(results["hgb"]["oof"] - y_dev, LGB_BEST["oof"] - y_dev)[0, 1]
print(f"LightGBM 최고와 오차 상관: {corr:.3f} (1에 가까우면 블렌딩 효과 작음)")

## STEP 9. 최종 조합
- 블렌딩 가중치 `w`(HGB 비중)를 OOF로 0.05 간격 탐색. OOF로 고른 가중치라 약간 낙관적 → 홀드아웃으로 확인
- 후보(LightGBM 최고, 블렌딩) 중 **기준 대비 채택 기준을 통과**한 것 가운데 OOF가 가장 낮은 것을 최종으로. 없으면 기준 유지

In [ ]:
ws = np.round(np.arange(0, 1.0001, 0.05), 2)
blend_scores = [rmse(y_dev, (1 - w) * LGB_BEST["oof"] + w * results["hgb"]["oof"]) for w in ws]
W_HGB = float(ws[int(np.argmin(blend_scores))])

blend_oof = (1 - W_HGB) * LGB_BEST["oof"] + W_HGB * results["hgb"]["oof"]
blend_fold = [rmse(y_dev[i_va], blend_oof[i_va]) for _, i_va in FOLDS]
results["blend"] = {"name": f"blend(w_hgb={W_HGB})", "kind": "blend", "oof": blend_oof,
                    "oof_rmse": rmse(y_dev, blend_oof), "fold_rmse": blend_fold, **seg_stats(blend_oof)}
print(f"블렌딩 최적 w_hgb = {W_HGB} | OOF {results['blend']['oof_rmse']:.4f}")

cands = [r for r in (LGB_BEST, results["blend"]) if r is not results["기준"] and passes(r, results["기준"])]
FINAL = min(cands, key=lambda r: r["oof_rmse"]) if cands else results["기준"]
if FINAL["kind"] == "blend" and W_HGB == 0:
    FINAL = LGB_BEST
print("최종:", FINAL["name"])

rows = []
for r in [results["기준"], LGB_BEST, results["hgb"], results["blend"]]:
    gain, folds_up = compare(r, results["기준"])
    rows.append({"모델": r["name"], "OOF RMSE": round(r["oof_rmse"], 4),
                 "기준 대비(음수=개선)": round(-gain, 4) + 0.0, "개선 fold": folds_up,
                 "fold RMSE": np.round(r["fold_rmse"], 4).tolist(),
                 "큰상승 편향": r["big_bias"], "상승시작 편향": r["onset_bias"],
                 "큰상승 SSE 비중": r["big_sse_share"], "나머지 RMSE": r["rest_rmse"]})
summary = pd.DataFrame(rows)
summary.to_csv(RESULT_DIR / "model_summary.csv", index=False, encoding="utf-8-sig")
summary

### 9-1. 사상별 오차 (기준 vs 최종)
오차 상위 사상만 좋아지고 나머지 사상이 나빠지는지 확인.

In [ ]:
ev = (dev.assign(base=results["기준"]["oof"], final=FINAL["oof"])
      .groupby("event_id")
      .apply(lambda d: pd.Series({"sse_기준": float(((d[TARGET] - d["base"]) ** 2).sum()),
                                  "rmse_기준": rmse(d[TARGET], d["base"]),
                                  "rmse_최종": rmse(d[TARGET], d["final"]), "n_rows": len(d)})))
ev["변화"] = ev["rmse_최종"] - ev["rmse_기준"]
ev = ev.sort_values("sse_기준", ascending=False)
ev.to_csv(RESULT_DIR / "event_rmse.csv", encoding="utf-8-sig")

top = ev.head(10)
print(f"상위 5개 사상 오차제곱합 비중(기준): {ev['sse_기준'].head(5).sum() / ev['sse_기준'].sum():.1%}")
print("나머지 사상 평균 RMSE 변화:", round(ev.iloc[10:]["변화"].mean(), 4),
      f"| 나빠진 사상 {int((ev['변화'] > 0).sum())}/{len(ev)}")
top.round(4)

## STEP 10. 홀드아웃 확인 (최종 조합 확정 후 **한 번만** 실행)
- dev 전체로 학습, LightGBM 반복 수는 CV best_iteration 평균 (홀드아웃으로 early stopping 하지 않음)
- 기준과 최종을 같은 조건으로 비교. 이 결과를 보고 파라미터를 다시 고르지 않음

In [ ]:
RUN_HOLDOUT = False   # 최종 조합 확정 후 True로 바꿔 한 번 실행

def fit_full(r, data):
    # 최종 학습용: 블렌딩이면 (LightGBM, HGB) 두 모델을 만들어 (모델, 가중치) 목록으로 반환
    parts = [(LGB_BEST, 1 - W_HGB), (results["hgb"], W_HGB)] if r["kind"] == "blend" else [(r, 1.0)]
    out = []
    for p, w in parts:
        X, yy = data[p["features"]], data[TARGET].to_numpy()
        if p["kind"] == "lgb":
            m = fit_lgb(p["params"], X, yy, n_round=int(np.mean(p["best_iters"])))
        else:
            m = fit_hgb(p["params"], X, yy)
        out.append((m, w, p["features"]))
    return out

def predict_full(models, data):
    return sum(w * predict(m, data[f]) for m, w, f in models)

if RUN_HOLDOUT and len(hold):
    hold_rows = []
    for r in (results["기준"], FINAL):
        p = predict_full(fit_full(r, dev), hold)
        hold_rows.append({"모델": r["name"], "홀드아웃 RMSE": round(rmse(hold[TARGET], p), 4),
                          "CV OOF RMSE": round(r["oof_rmse"], 4)})
    hold_summary = pd.DataFrame(hold_rows)
    hold_summary.to_csv(RESULT_DIR / "holdout_check.csv", index=False, encoding="utf-8-sig")
    display(hold_summary)

## STEP 11. Test 예측 · 제출 파일 생성
- 홀드아웃 확인 뒤 **train 전체(dev + 홀드아웃)** 로 다시 학습해서 예측. 반복 수는 CV 평균 그대로
- 제출 양식 검증은 베이스라인과 동일 (행 수·row_id 순서·결측)

In [ ]:
MAKE_SUBMISSION = False   # 홀드아웃 확인 후 True

if MAKE_SUBMISSION:
    full = pd.concat([dev, hold], ignore_index=True)
    te["pred"] = predict_full(fit_full(FINAL, full), te)

    submission = sub[["row_id"]].merge(te[["row_id", "pred"]], on="row_id", how="left")
    submission = submission.rename(columns={"pred": "pred_maxrise_6h"})

    assert len(submission) == len(sub), "행 수 불일치"
    assert submission["row_id"].equals(sub["row_id"]), "row_id 순서 불일치"
    assert submission["pred_maxrise_6h"].notna().all(), "결측 예측값 존재"

    out_path = RESULT_DIR / f"submission_{FINAL['kind']}_cv{FINAL['oof_rmse']:.4f}.csv"
    submission.to_csv(out_path, index=False, encoding="utf-8")
    print("저장:", out_path)
    display(submission.describe())

## STEP 12. 최종 설정 저장 (재혁 통합 코드용)

In [ ]:
final_model = {
    "final": FINAL["name"],
    "kind": FINAL["kind"],
    "features_file": str(FEATURE_FILE.relative_to(ROOT)),
    "lgb_params": LGB_BEST["params"],
    "lgb_n_round": int(np.mean(LGB_BEST["best_iters"])),
    "hgb_params": HGB_PARAMS,
    "w_hgb": W_HGB if FINAL["kind"] == "blend" else 0.0,
    "cv_oof_rmse": round(FINAL["oof_rmse"], 4),
    "base_oof_rmse": round(results["기준"]["oof_rmse"], 4),
}
with open(RESULT_DIR / "final_model.json", "w", encoding="utf-8") as f:
    json.dump(final_model, f, ensure_ascii=False, indent=2)
print(json.dumps(final_model, ensure_ascii=False, indent=2))
print(sorted(p.name for p in RESULT_DIR.iterdir()))